# شروع کار با قسمت اول PyDara — نمونه بدون اینترنت

تمام پاسخ‌ها، قیمت‌ها و برخی شناسه‌های این دفترچه **مصنوعی و صرفاً برای آموزش و آزمون** هستند؛ این نمونه قیمت روز بازار نیست. ابتدا بسته محلی را طبق README نصب و کرنل را Restart کنید. این دفترچه هیچ درخواستی به بورس نمی‌فرستد. ذخیره نمونه در پوشه `pydara_demo_cache` است و داده مصنوعی وارد فضای کش واقعی نمی‌شود.

سلول‌ها را به ترتیب اجرا کنید. برای دریافت واقعی از دفترچه دوم استفاده کنید.

## سلول ۱ — ورود کتابخانه‌ها و بررسی محیط

In [ ]:
import sys
import pydara
from pathlib import Path
from IPython.display import display
from pydara.tse import FetchPolicy, FreshnessPolicy
from pydara.tse.samples import sample_client, sample_async_client

print("Python:", sys.version)
print("PyDara:", pydara.__version__)
print("SYNTHETIC SAMPLE ONLY - NOT MARKET DATA")

## سلول ۲ — تابع نمایش نتیجه

In [ ]:
def show_quote(quote):
    print("Symbol:", quote.instrument.symbol)
    print("Last:", quote.last_price, quote.instrument.currency)
    print("Closing:", quote.closing_price)
    print("Source time:", quote.source_timestamp)
    print("Fetched at:", quote.retrieved_at)
    print("Flags:", quote.meta.quality_flags)
    display(quote.to_frame())

## سلول ۳ — تنظیمات ثابت

In [ ]:
SYMBOL = 'فولاد'
DATA_DIR = Path.cwd() / "pydara_demo_cache"
OFFLINE_POLICY = FetchPolicy(freshness=FreshnessPolicy(allow_stale=True))

## سلول ۴ — اجرای دریافت

In [ ]:
with sample_client(storage=True, data_dir=DATA_DIR) as client:
    quote = client.get_quote(SYMBOL)
    saved = client.flush()
    assert saved.complete
    status = client.storage_status()

## سلول ۵ — نمایش گزارش و وضعیت ذخیره

In [ ]:
show_quote(quote)
print("Storage:", status)
assert str(quote.last_price) == "1234.5"

## بازیابی آفلاین همان پاسخ ذخیره‌شده

In [ ]:
with sample_client(storage=True, data_dir=DATA_DIR, mode="offline") as client:
    cached = client.get_quote(SYMBOL, policy=OFFLINE_POLICY)
    assert client.request_count == 0

print("CACHED SYNTHETIC DATA, NOT LIVE")
show_quote(cached)

## دریافت گروهی با حفظ خطای هر ورودی

In [ ]:
with sample_client() as client:
    result = client.get_quotes(['فولاد', 'ناموجود', 'فولاد'], on_error="collect")

display(result[["request_index", "input_symbol", "status", "last_price", "error_code"]])

## اجرای همان دریافت با رابط Async

In [ ]:
async with sample_async_client() as client:
    async_quote = await client.get_quote(SYMBOL)

assert async_quote.last_price == quote.last_price
show_quote(async_quote)